# AGENTS_030 — Multi-Agent Infrastructure Triage (MI300 / vLLM)

Runs **change-impact-agent** (pre-merge) and **log-analysis-agent** (post-CI) as separate notebook steps.

| Step | Agent | Demo asset |
|------|-------|------------|
| 5 | change-impact-agent | PR **5572** (sample briefing) |
| 6 | log-analysis-agent | GHA run **[27710372755](https://github.com/ROCm/TheRock/actions/runs/27710372755)** job `81992436725` (rocSPARSE OOM) |

## Steps
1. Launch vLLM (terminal)
2. Config + verify vLLM
3. Install deps
4. *(commented out)* combined direct smoke test — use Steps 5–6 instead
5. **change-impact-agent** — pre-merge blast radius / rollout
6. **log-analysis-agent** — post-CI log triage + vLLM executive summary


## Step 1: Launch vLLM

```bash
VLLM_USE_TRITON_FLASH_ATTN=0 \
vllm serve Qwen/Qwen3-30B-A3B \
  --served-model-name Qwen3-30B-A3B \
  --api-key abc-123 \
  --port 8000 \
  --trust-remote-code
```

For optional Pydantic orchestrator later, add: `--enable-auto-tool-choice --tool-call-parser hermes`

Monitor: `watch rocm-smi`


In [1]:
import os
import sys
from pathlib import Path


def _candidate_search_roots() -> list[Path]:
    """Roots to search when Path.cwd() is broken (deleted cwd / MI300 notebooks)."""
    roots: list[Path] = []
    env_agents = os.environ.get("THEROCK_AGENTS_DIR")
    if env_agents:
        roots.append(Path(env_agents))
    for fixed in (
        "/workspace/TheRock-old/agents",
        "/workspace/TheRock/agents",
        Path.home() / "TheRock-old" / "agents",
        Path.home() / "TheRock" / "agents",
    ):
        roots.append(Path(fixed))
    for key in ("PWD", "JUPYTER_SERVER_ROOT", "THEROCK_ROOT"):
        val = os.environ.get(key)
        if val:
            roots.append(Path(val))
    try:
        from IPython import get_ipython

        ip = get_ipython()
        if ip is not None:
            cfg = getattr(ip, "config", None)
            if cfg:
                root = cfg.get("ServerApp", {}).get("root_dir")
                if root:
                    roots.append(Path(root))
    except Exception:
        pass
    try:
        roots.append(Path.cwd().resolve())
    except FileNotFoundError:
        pass
    # De-dupe while preserving order
    seen: set[str] = set()
    unique: list[Path] = []
    for root in roots:
        key = str(root)
        if key in seen:
            continue
        seen.add(key)
        unique.append(root)
    return unique


def resolve_agents_dir() -> Path:
    for start in _candidate_search_roots():
        try:
            resolved_start = start.resolve()
        except OSError:
            continue
        for candidate in [resolved_start, *resolved_start.parents]:
            if (candidate / "multi_agent_tools.py").is_file():
                return candidate
            if candidate.name == "notebook" and (candidate.parent / "multi_agent_tools.py").is_file():
                return candidate.parent
            if (candidate / "agents" / "multi_agent_tools.py").is_file():
                return candidate / "agents"
    raise RuntimeError(
        "Could not find agents/multi_agent_tools.py. Set THEROCK_AGENTS_DIR=/path/to/agents "
        "or start Jupyter from TheRock-old/agents/notebook/"
    )


AGENTS_DIR = resolve_agents_dir()
THEROCK_ROOT = AGENTS_DIR.parent if AGENTS_DIR.name == "agents" else AGENTS_DIR
NOTEBOOK_OUT = AGENTS_DIR / "notebook" / "out"

# Repair broken kernel cwd (e.g. after deleting sample-runs while cwd was inside it).
try:
    Path.cwd()
except FileNotFoundError:
    recover = AGENTS_DIR / "notebook" if (AGENTS_DIR / "notebook").is_dir() else AGENTS_DIR
    os.chdir(recover)
    print("Recovered kernel cwd →", recover)

if str(AGENTS_DIR) not in sys.path:
    sys.path.insert(0, str(AGENTS_DIR))

from multi_agent_tools import (
    DEFAULT_DEMO_LOG,
    DEFAULT_DEMO_LOG_URL,
    DEFAULT_DEMO_JOB_ID,
    DEFAULT_DEMO_PR,
    DEFAULT_DEMO_RUN_ID,
    DEMO_PRS,
)

BASE_URL = os.environ.get("VLLM_BASE_URL", os.environ.get("BASE_URL", "http://localhost:8000/v1"))
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "abc-123")
LLM_MODEL = os.environ.get("VLLM_MODEL", os.environ.get("LLM_MODEL", "Qwen3-30B-A3B"))

# MI300 notebook: vLLM executive summaries for log-analysis-agent (errors stay tool-only)
USE_VLLM = True

os.environ["BASE_URL"] = BASE_URL
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["LLM_MODEL"] = LLM_MODEL

if USE_VLLM:
    os.environ["USE_VLLM"] = "1"
    os.environ["USE_VLLM_SUMMARY"] = "1"
    os.environ["LOG_SUMMARY_BACKEND"] = "vllm"
    os.environ.setdefault("VLLM_BASE_URL", BASE_URL)
    os.environ.setdefault("VLLM_MODEL", LLM_MODEL)

print("AGENTS_DIR   =", AGENTS_DIR)
print("THEROCK_ROOT =", THEROCK_ROOT)
print("USE_VLLM     =", USE_VLLM)
print("BASE_URL     =", BASE_URL)
print("LLM_MODEL    =", LLM_MODEL)
print("DEMO_PRS     =", DEMO_PRS)
print("DEMO_PR      =", DEFAULT_DEMO_PR)
print("DEMO_RUN_ID  =", DEFAULT_DEMO_RUN_ID)
print("DEMO_JOB_ID  =", DEFAULT_DEMO_JOB_ID)
print("DEFAULT_LOG  =", DEFAULT_DEMO_LOG)
print("RUN URL      =", DEFAULT_DEMO_LOG_URL)

# Fail fast if MI300 checkout is behind (reads file on disk — no stale import cache).
_mat_py = AGENTS_DIR / "multi_agent_tools.py"
_required_api = ("run_change_impact_for_demo_pr", "run_log_analysis_for_demo_run")
_missing_api = [n for n in _required_api if f"def {n}" not in _mat_py.read_text(encoding="utf-8")]
if _missing_api:
    raise RuntimeError(
        f"Stale {_mat_py} — missing: {', '.join(_missing_api)}.\n"
        "Terminal:\n"
        "  cd /workspace/TheRock-old\n"
        "  git fetch origin && git checkout feature/change-impact-agent\n"
        "  git pull origin feature/change-impact-agent\n"
        "  git log -1 --oneline   # c9c176c3a+\n"
        "Then Kernel → Restart and re-run from Step 2."
    )
print("multi_agent_tools API OK (live analysis Steps 5–6)")


AGENTS_DIR   = /workspace/TheRock-old/agents
THEROCK_ROOT = /workspace/TheRock-old
USE_VLLM     = True
BASE_URL     = http://localhost:8000/v1
LLM_MODEL    = Qwen3-30B-A3B
DEMO_PRS     = (5572, 5688, 5480, 5718)
DEMO_PR      = 5572
DEMO_RUN_ID  = 27710372755
DEMO_JOB_ID  = 81992436725
DEFAULT_LOG  = /workspace/TheRock-old/agents/log-analysis-agent/sample-runs/run-27710372755/job-81992436725/job-81992436725.log
RUN URL      = https://github.com/ROCm/TheRock/actions/runs/27710372755
multi_agent_tools API OK (live analysis Steps 5–6)


In [2]:
import httpx

VLLM_REACHABLE = False
headers = {"Authorization": f"Bearer {OPENAI_API_KEY}"}

try:
    r = httpx.get(f"{BASE_URL}/models", headers=headers, timeout=15.0)
    VLLM_REACHABLE = r.status_code < 400
    if VLLM_REACHABLE:
        print("vLLM reachable:", [m.get("id") for m in r.json().get("data", [])[:3]])
    else:
        print("vLLM models endpoint:", r.status_code, r.text[:200])
except Exception as exc:
    print("vLLM not reachable:", exc)
    print("Start vLLM in Step 1. Step 5 still works; Step 6 needs vLLM for executive summary.")


vLLM reachable: ['Qwen3-30B-A3B']


In [3]:
import sys
!{sys.executable} -m pip install -q -r "{AGENTS_DIR / 'requirements-notebook.txt'}"



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python3.12 -m pip install --upgrade pip


## Step 4: Direct combined smoke test *(commented out)*

Skipped in the MI300 / vLLM workflow — run **Step 5** (change-impact) and **Step 6** (log-analysis) instead.


In [4]:
# # Step 4 — combined direct smoke test (offline / no vLLM orchestrator)
# from multi_agent_tools import (
#     list_demo_assets,
#     run_change_impact_for_pr,
#     run_log_analysis_for_path,
#     run_infrastructure_triage_loop,
# )
#
# print(list_demo_assets())
# print(run_change_impact_for_pr(DEFAULT_DEMO_PR))
# print(run_log_analysis_for_path(str(DEFAULT_DEMO_LOG), use_vllm_summary=USE_VLLM))
# print(run_infrastructure_triage_loop(DEFAULT_DEMO_PR, str(DEFAULT_DEMO_LOG), use_vllm_summary=USE_VLLM))


## Step 5: change-impact-agent (pre-merge)

Re-runs the **deterministic analysis pipeline** on PR **5572** when git refs / GitHub token are available:

`manifest_diff` → `path_diff` → `content_diff` → `impact_graph` → `ci_mapping` → `rollout_strategy`

1. **Analyze:** `analyze.build_report()` (live) — not just loading `sample-runs/pr-5572/report.json`
2. **Output:** `agents/notebook/out/pr-5572/report.json` + `report.html`
3. **Then vLLM:** `executive_summary.md` when `USE_VLLM=True` (Step 2)

Falls back to committed `sample-runs/` only if PR refs cannot be resolved.


In [5]:
import importlib
import multi_agent_tools

importlib.reload(multi_agent_tools)
multi_agent_tools.check_demo_notebook_api()

change_impact_summary = multi_agent_tools.run_change_impact_for_demo_pr(
    DEFAULT_DEMO_PR,
    use_vllm_summary=USE_VLLM,
)
print(change_impact_summary)


=== change-impact-agent: pre-merge analysis ===
PR #5572 — mode: sample_fallback
Pipeline: manifest_diff → path_diff → content_diff → impact_graph → ci_mapping
Output: notebook/out/pr-5572/report.json

PR #5572 change impact
Severity: MEDIUM (blast radius 40/100)
Rollout: Canary gfx family + component-specific test labels
Suggested labels: test:miopen, test_filter:quick
Changed files: 1
Rationale:
  - CI test matrix or GitHub Actions configuration changed
  - GHA wrapper timeout for `miopen`: 60 → 120 minutes (per-test limits still from test_categories.yaml)

Note: could not resolve git refs or fetch PR — showing committed sample-runs JSON.
Set GITHUB_TOKEN in change-impact-agent/.env and ensure pr-* ref exists to re-analyze live.

=== vLLM reviewer brief (after analysis JSON) ===
# Change Impact Executive Summary

**Range:** `7b7b238e` → `c70f211c`
**Severity:** MEDIUM (blast radius 40/100)

## What changed

### Changed files (git diff)
- `build_tools/github_actions/fetch_test_configu

## Step 6: log-analysis-agent (post-CI)

Re-runs **full tool_only analysis** (grep + KB → fresh `report.json`) on all bundled failed jobs for GHA run **[27710372755](https://github.com/ROCm/TheRock/actions/runs/27710372755)**.

1. **Input:** raw `.log` files only from `log-analysis-agent/sample-runs/run-27710372755/` (not pre-baked reports)
2. **Analyze:** `build_report()` — ERROR/FATAL, `hipErrorOutOfMemory`, gtest `[FAILED]`, KB lookup
3. **Output:** `agents/notebook/out/run-27710372755/job-*/report.json` + `run_summary.json`
4. **Then vLLM** on **every** failed job: `executive_summary.md` per job from ranked JSON

Job `81992436725` (rocSPARSE OOM) is the demo focus, but all 8 bundled jobs get vLLM briefs when `USE_VLLM=True`.


In [6]:
import importlib
import multi_agent_tools

importlib.reload(multi_agent_tools)
multi_agent_tools.check_demo_notebook_api()

log_analysis_summary = multi_agent_tools.run_log_analysis_for_demo_run(
    preset="therock_multi_arch",
    use_vllm_summary=USE_VLLM,
)
print(log_analysis_summary)


=== log-analysis-agent: tool_only pass (grep + KB) ===
Run 27710372755 — re-analyzed 8 bundled job log(s)
Input logs: log-analysis-agent/sample-runs/run-27710372755/*.log
Fresh output: notebook/out/run-27710372755/

Rollup:
  job 81975007413 (Linux::release / Build Multi-Arch Stages / comm-libs / Stage): 2 errors | primary: 2026-06-17T20:36:48.3343455Z ##[error]The operation was canceled.
  job 81975007686 (Linux::release / Build Multi-Arch Stages / math-libs (gfx94X): 2 errors | primary: 2026-06-17T20:36:48.5684534Z ##[error]The operation was canceled.
  job 81992436676 (Windows::release / Test gfx110X-all / Test rocblas / Test ro): 2 errors | primary: 2026-06-17T20:36:56.0158814Z ##[error]The operation was canceled.
  job 81992436725 [demo focus — rocSPARSE OOM] (Windows::release / Test gfx110X-all / Test rocsparse / Test ): 46 errors | primary: 2026-06-17T20:19:38.4851735Z 1: //                            "msg"     : "prior to hipLau
  job 81992436788 (Windows::release / Test gfx110

## Outputs

| Step | Artifact |
|------|----------|
| 5 | `agents/notebook/out/pr-5572/report.json` + `executive_summary.md` |
| 6 | `agents/notebook/out/run-27710372755/run_summary.json` |
| 6 | `agents/notebook/out/run-27710372755/job-81992436725/report.json` |
| 6 | `agents/notebook/out/run-27710372755/job-*/executive_summary.md` (all jobs) |

Bundled **log inputs** live under `log-analysis-agent/sample-runs/run-27710372755/`. Step 6 always regenerates analysis under `notebook/out/`.
